# Traffic data analysis: practice 1

Two tasks and their existing solutions, organized for Jupyter Notebook. The original Russian task statements are preserved below. Run the cells in order.

The original calculations are retained; this edition organizes the code and makes file paths portable.


# Цель задания

Изучить основы работы с данными с применением Python и Jupyter Notebook

# Рекомендованная литература

- "Learning Pandas 2.0"  Matthew Rosch
- "Python и анализ данных" Уэс Маккинни


## Setup

Install the packages from `requirements.txt` before opening this notebook. The path setup supports launching Jupyter from the repository root or the `notebooks` folder. Generated tables are written to `results/`.


In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import DataFrame, Series 
import math
pd.options.mode.chained_assignment = None
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "all_tracks").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "all_tracks"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks folder.")
OUTPUT_DIR = ROOT / "results"
OUTPUT_DIR.mkdir(exist_ok=True)


## Load the example dataset

The example uses `all_tracks/125.B16.DT1.json`. Change `filename` to select another supplied JSON file.


In [ ]:
filename = DATA_DIR / "125.B16.DT1.json"
with filename.open(encoding="utf-8-sig") as f:
    data = json.load(f)

df = pd.json_normalize(data["objects"])
df = df.explode("rows_data")
df = pd.DataFrame(df["rows_data"].tolist())


## Задача 1
Проанализируйте полученные данные и преобразуйте их в удобный для анализа dataset
(пример преобразования приведен ниже)

Проведите анализ данных. Опишите, что вы смогли узнать про эти данные.
Есть ли в этих данных выбросы?

### Existing outlier checks


In [ ]:
parametrs = ['obj_width', 'obj_speed', 'obj_length']

for i in parametrs:
    df= df.sort_values(by=i)
    IQR = df[i][math.floor(len(df[i])*0.75)] - df[i][math.floor(len(df[i])*0.25)]
    IQR *= 1.5
    for q in range(len(df[i]) -1, -1, -1):
        if  df[i][q] < df[i][math.floor(len(df[i])*0.25)] - IQR or df[i][q] > df[i][math.floor(len(df[i])*0.25)] + IQR:
            pass
            #print(i, df[i][q], df[i][math.floor(len(df[i])*0.75)], df[i][math.floor(len(df[i])*0.25)]) 

for i in df['obj_width']:
    if i < 1.7 or i > 2.1:
        print('obj_width', i)

for i in df['obj_length']:
    if i < 3 or i > 12:
        print('obj_width', i)

for i in df['obj_speed']:
    if i < 0 or i > 250:
        print('obj_width', i)

for i in df['heading']:
    if 10 < i < 170 or 190 < i < 350 :
        print('heading', i)


### Dataset structure and descriptive statistics


In [ ]:
df.info()
#df.head()


In [ ]:
df.describe()

## Задача 2
Определить следующие параметры потока: 
    средняя скорость, 
    средняя скорость по полосам,
    V85  (85-й перцентиль),
    V85 по полосам

### *дополнительно для вдумчивых 
среднее и минимальное расстояние между авто по полосам 

###### Следует обратить внимание, что машинки едут одна за другой и для этой задачи придется учитывать время. Obj_id не является уникальным папаметром, повторяется циклично от 0 до 255

### Data records and lane statistics


In [ ]:
from datetime import datetime
import os
import json

class DataObject:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)
def to_custom_object(pandas_row):
    return DataObject(**pandas_row._asdict())

class Lane:
    Alllanes = {}
    def __init__(self, car):
        if list(Lane.Alllanes.keys()).count(car.lane) :
            Lane.Alllanes[car.lane].cars.append(car)
        else: 
            self.cars = [car]
            Lane.Alllanes[car.lane] = self
            if car.lane < 3:
                self.np = 0
            else:
                self.np = 1

            
                                    
    @classmethod
    def calculate(cls):
        
        for i in list(Lane.Alllanes.keys()):
            Lane.Alllanes[i].carspeed = []
            for q in Lane.Alllanes[i].cars:  
                Lane.Alllanes[i].carspeed.append(q.averagespeed)
            Lane.Alllanes[i].carspeed.sort()
            Lane.Alllanes[i].averagespeed = sum(Lane.Alllanes[i].carspeed) / len(Lane.Alllanes[i].carspeed)
            Lane.Alllanes[i].V85 = Lane.Alllanes[i].carspeed[math.floor(len(Lane.Alllanes[i].carspeed)*0.85)]
        
        for i in list(Lane.Alllanes.keys()):
            logslines = []
            Lane.Alllanes[i].intervals = {}
            for q in Lane.Alllanes[i].cars:
                logslines.extend(q.lanes)
            logslines.sort(key=lambda x: x.point_x)
            logslines.sort(key=lambda x: x.timestamp)

            for q in range(1, len(logslines)):
                u_ids = [logslines[q-1].u_id, logslines[q].u_id]
                u_ids.sort()
                obj_ids = [logslines[q-1].obj_id, logslines[q].obj_id]
                obj_ids.sort()
                                                
                time = logslines[q].timestamp - logslines[q-1].timestamp
                if u_ids[0] != u_ids[1] and time == 0 and str(logslines[q].point_x) != 'nan' and str(logslines[q-1].point_x) != 'nan' and str(logslines[q].point_y) != 'nan' and str(logslines[q-1].point_y) != 'nan' and 1 > abs(logslines[q].point_y - logslines[q-1].point_y):
                    if not list(Lane.Alllanes[i].intervals.keys()).count(str(u_ids)):
                        Lane.Alllanes[i].intervals[str(u_ids)] = []
                    addway = logslines[q-1].obj_speed / 3.6 * time
                    if 90 < logslines[q-1].heading < 270:
                        Lane.Alllanes[i].intervals[str(u_ids)].append(abs(logslines[q-1].point_x - addway - logslines[q].point_x))
                    else:
                        Lane.Alllanes[i].intervals[str(u_ids)].append(abs(logslines[q-1].point_x + addway - logslines[q].point_x))
                    Lane.Alllanes[i].intervals[str(u_ids)][-1] -= (CAR(u_ids[0]).length + CAR(u_ids[1]).length)/2
                    Print.protocol[logslines[q].timestamp]['distance'].append({'car_ids' : obj_ids, 'length' : Lane.Alllanes[i].intervals[str(u_ids)][-1]})
                    
                    #if Lane.Alllanes[i].intervals[str(u_ids)][-1] < 1:
                     #   print(logslines[q-1].point_y, logslines[q-1].obj_id, logslines[q].point_y, logslines[q].obj_id)
            Lane.Alllanes[i].averageintervals = []
            Lane.Alllanes[i].aintervals = []
            for q in list(Lane.Alllanes[i].intervals.keys()):
                Lane.Alllanes[i].averageintervals.append(sum(Lane.Alllanes[i].intervals[q]) / len(Lane.Alllanes[i].intervals[q]))
                Lane.Alllanes[i].aintervals.extend(Lane.Alllanes[i].intervals[q])

            Lane.Alllanes[i].aintervals.sort()
            if Lane.Alllanes[i].averageintervals:
                Lane.Alllanes[i].averageinterval = sum(Lane.Alllanes[i].averageintervals) / len(Lane.Alllanes[i].averageintervals)
                Lane.Alllanes[i].maxinterval = Lane.Alllanes[i].aintervals[math.floor(len(Lane.Alllanes[i].aintervals) * 0.05)]
            else:
                Lane.Alllanes[i].averageinterval = -1
                Lane.Alllanes[i].maxinterval = -1
                                    
                #comparedistance(Lane.Alllanes[i].cars[q-1].lanes, Lane.Alllanes[i].cars[q].lanes)


### Vehicle statistics and tracking


In [ ]:
class BFC():
    @classmethod

    def calculate(cls):
        cls.carspeed = []
        for i in cls.Allcars:
            i.lanes.sort(key=lambda x: x.obj_length)
            i.length = i.lanes[len(i.lanes)//2].obj_length
        for i in cls.Allcars:
            i.lanes.sort(key=lambda x: x.timestamp)

            for n in range(0, len(i.lanes), 1):
                if hasattr(i.lanes[n], 'point_x'):
                    n = [i.lanes[n].point_x, i.lanes[n].timestamp]
                    break
            for f in range(len(i.lanes)-1, -1, -1):
                if hasattr(i.lanes[f], 'point_x'):
                    f = [i.lanes[f].point_x, i.lanes[f].timestamp]
                    break
            if f[1] - n[1] and n < f:
                i.averagespeed = abs(n[0] - f[0])  / (f[1] - n[1]) * 3.6
            else:
                i.averagespeed = i.lanes[0].obj_speed            
            cls.carspeed.append(i.averagespeed)

        #print(cls.carspeed)
        cls.carspeed.sort()
        cls.averagespeed = sum(cls.carspeed) / len(cls.carspeed)
        cls.V85 = cls.carspeed[math.floor(len(cls.carspeed)*0.85)]
        
class CarLane(BFC):
    Allcars = []
    def __init__(self, lane):
        self.lanes = [lane]
        self.lane = lane.lane
        CarLane.Allcars.append(self) 
        self.u_id = lane.u_id       

def CAR(uid):
    for i in Car.Allcars:
        if i.u_id == uid:
            return i
class Car(BFC):
    Allcars = []
    Switches = 0
    def __init__(self, lane):
        A = [obj for obj in Car.Allcars if obj.obj_id == lane.obj_id]
        F = 1
        for i in A:
            if abs(lane.timestamp - i.ltime) < 60:
                i.ltime = max(lane.timestamp, i.ltime)
                lane.u_id = i.u_id
                F = 0
                i.lanes.append(lane)
        if F:
            self.u_id = len(Car.Allcars)  
            lane.u_id = self.u_id
            self.ltime = lane.timestamp
            self.obj_id = lane.obj_id     
            self.lanes = [lane]  
            if 90 < lane.heading < 270:
                self.np = 0
            else:
                self.np = 1


            Car.Allcars.append(self)
    def linesculc():
        for i in Car.Allcars:
            i.carlane = [CarLane(i.lanes[0])]
            Lane(i.carlane[-1])
            for q in i.lanes:
                if q.lane != i.carlane[-1].lane:
                    i.carlane.append(CarLane(q))
                    Lane(i.carlane[-1])
                    Car.Switches += 1
                else:
                    i.carlane[-1].lanes.append(q)



    def noexteral():
        logs = [[], []]
        for x in Car.Allcars:
            for y in x.lanes:
                if x.np == 1:
                    logs[1].append(y.point_x)
                else:
                    logs[0].append(y.point_x)
        logs[0].sort()
        logs[1].sort()

        for q in range(len(Car.Allcars) -1, -1, -1):
            Car.Allcars[q].lanes.sort(key=lambda x: x.point_x)

                #print(len(Car.Allcars[q].lanes), logs[math.floor(len(logs)*0.75)] + IQR, logs[math.floor(len(logs)*0.25)] - IQR, Car.Allcars[q].obj_id, Print.nm)
            if len(Car.Allcars[q].lanes) < 10 or Car.Allcars[q].lanes[0].point_x > logs[Car.Allcars[q].np][math.floor(len(logs[Car.Allcars[q].np])*0.25)] or Car.Allcars[q].lanes[-1].point_x < logs[Car.Allcars[q].np][math.floor(len(logs[Car.Allcars[q].np])*0.75)]:
                #print(len(Car.Allcars[q].lanes))
                Car.Allcars.pop(q)
        #print(logs)
        #print(len(Car.Allcars), Print.nm)
            


    '''@property
    def lanes(cls):
        lanes = []
        for i in self.carlane:
            lanes.extend(i.lanes)
        return lanes          '''


### Calculations and summary tables


In [ ]:
class Calculate():
    def __init__(self):
        Car.noexteral()
        Car.linesculc()        
        Car.calculate()
        CarLane.calculate()
        Lane.calculate()

class Print():
    filename = ''
    dt = ''
    nm = ''
    d = {}
    tables = {'unic_cars' : {}, 'intensity_cars' : {}, 'average_speed_cars' : {}, 'med_speed_cars' : {}, 'V85_speed_cars' : {}, 'number_distance' : {},  'average_distance' : {},  'min_distance' : {},
              'cars_speed' : {}, 'cars_distance' : {}, 'swiches_car' : {}}
    def __init__(self):
        df = Print.dt
        time = df['time'][len(df['time'])//2]
        time = df['timestamp'][len(df['time'])//2]
        tm = df['timestamp'][len(df['time'])-1] - df['timestamp'][0]  
        for i in list(Print.tables.keys()):
            Print.tables[i][time] = {}
        Print.d[time] = Print.nm
        #print(df['time'][len(df['time'])//2], Print.filename)
        #print('Всего уникальных машин:', len(Car.Allcars))
        Print.tables['unic_cars'][time]['all'] = len(Car.Allcars)
        #print('Всего перестроений:', Car.Switches)
        Print.tables['swiches_car'][time]['all'] = Car.Switches
        #print('Средняя скорость машин:', round(Car.averagespeed, 2))
        Print.tables['average_speed_cars'][time]['all'] = round(Car.averagespeed, 2)
        #print('Медианная скорость машин:', round(Car.carspeed[math.floor(len(Car.carspeed)*0.5)], 2))        
        Print.tables['med_speed_cars'][time]['all'] = round(Car.carspeed[math.floor(len(Car.carspeed)*0.5)], 2)
        #print('Скорость по V85 машин:', round(Car.V85, 2))
        Print.tables['V85_speed_cars'][time]['all'] = round(Car.V85, 2)
        #print('Интенсивность машин:', round(len(Car.Allcars) / tm * 60, 2), 'машин/мин')
        Print.tables['intensity_cars'][time]['all'] = round(len(Car.Allcars)/ tm * 60, 2)

        Car.Intervals = []
        for x in list(Lane.Alllanes.keys()):
            Car.Intervals.extend(Lane.Alllanes[x].averageintervals)
        Car.Intervals.sort()
        print(Car.Intervals)
        if len(Car.Intervals):
            Print.tables['average_distance'][time]['all'] = sum(Car.Intervals) / len(Car.Intervals)
            #print('Среднее расстояние', Print.tables['average_distance'][time]['all'] )
            Print.tables['min_distance'][time]['all'] = round(Car.Intervals[math.floor(len(Car.Intervals)*0.05)], 2)
            n = 0
            for i in Car.Intervals:
                if i < 30:
                    n += 1
            Print.tables['number_distance'][time]['all<30'] = n
            Print.tables['number_distance'][time]['all'] = len(Car.Intervals)
            #print('Минимальное расстояние (5-й персентиль)', Print.tables['min_distance'][time]['all'] )
        A = list(Lane.Alllanes.keys())
        A.sort()
        NP = -1
        NPC = {'u_car' : [set(), set()], 'speed' : [[], []]}

        for i in A:
            if Lane.Alllanes[i].np == 0:
                [NPC['u_car'][0].add(x.u_id) for x in Lane.Alllanes[i].cars]
            else:
                [NPC['u_car'][1].add(x.u_id) for x in Lane.Alllanes[i].cars]

        [NPC['speed'][0].append(CAR(x).averagespeed) for x in NPC['u_car'][0]]
        [NPC['speed'][1].append(CAR(x).averagespeed) for x in NPC['u_car'][1]]
        NPC['speed'][0].sort()
        NPC['speed'][1].sort()
                        
        def T(n):
            if n == 1:
                return '1'
            else:
                return '-1'        
        for i in A:
        
            if(NP != Lane.Alllanes[i].np):
                NP = Lane.Alllanes[i].np
                #print('===============================')
                #if(NP == 1):
                    #print('Машин в прямом направлении', len(NPC['u_car'][1]))
                #else:
                    #print('Машин в обратном направлении', len(NPC['u_car'][0]))
                
                Print.tables['unic_cars'][time]['N  ' + T(NP)] = len(NPC['u_car'][NP])
                Print.tables['intensity_cars'][time]['N  ' + T(NP)] = len(NPC['u_car'][NP]) *60 / tm
                if len(NPC['speed'][NP]):
                    Print.tables['average_speed_cars'][time]['N  ' + T(NP)] = sum(NPC['speed'][NP]) / len(NPC['speed'][NP])
                    Print.tables['med_speed_cars'][time]['N  ' + T(NP)] = round(NPC['speed'][NP][math.floor(len(NPC['speed'][NP])*0.5)], 2)
                    Print.tables['V85_speed_cars'][time]['N  ' + T(NP)] = round(NPC['speed'][NP][math.floor(len(NPC['speed'][NP])*0.85)], 2)

            #print('===============================')
            #print('Полоса', i)
            #print('Уникальных машин:', len(Lane.Alllanes[i].cars))
            uc = set([x.u_id for x in Lane.Alllanes[i].cars])
            Print.tables['unic_cars'][time][i] = len(uc)
            #print('Интенсивность машин:', round(len(Lane.Alllanes[i].cars) / tm * 60, 2), 'машин/мин')
            #Print.tables['intensity_cars'][time][i] = round(len(Lane.Alllanes[i].cars) / tm * 60, 2)
            #print('Средняя скорость', round(Lane.Alllanes[i].averagespeed,2))
            Print.tables['average_speed_cars'][time][i] = round(Lane.Alllanes[i].averagespeed,2)
            #print('Медианная скорость машин:', round(Lane.Alllanes[i].carspeed[math.floor(len(Lane.Alllanes[i].carspeed)*0.5)], 2))     
            Print.tables['med_speed_cars'][time][i] = round(Lane.Alllanes[i].carspeed[math.floor(len(Lane.Alllanes[i].carspeed)*0.5)], 2)
            #print('Скорость по V85:', round(Lane.Alllanes[i].V85,2))
            Print.tables['V85_speed_cars'][time][i] = round(Lane.Alllanes[i].V85,2)

            Print.tables['cars_speed'][time][i] = Lane.Alllanes[i].carspeed
            if len(list(Lane.Alllanes[i].intervals.keys())):
                #print('Всего расстояний', len(list(Lane.Alllanes[i].intervals.keys())))
                Print.tables['number_distance'][time][i] = len(list(Lane.Alllanes[i].intervals.keys()))
                #print('Среднее растояние', round(Lane.Alllanes[i].averageinterval,2))
                Print.tables['average_distance'][time][i] = round(Lane.Alllanes[i].averageinterval,2)
                #print('Минимальное расстояние (5-й персентиль)', round(Lane.Alllanes[i].maxinterval,2))
                Print.tables['min_distance'][time][i] = round(Lane.Alllanes[i].maxinterval,2)
                Lane.Alllanes[i].averageintervals.sort()
                Print.tables['cars_distance'][time][0] = Car.Intervals #Lane.Alllanes[i].averageintervals
            Print.tables


### Dataset processing and export


In [ ]:
def start(name, output_dir):
    files = ['102.B2C6.DT1.json', '103.B2C7.DT1.json', '104.B1C7.DT1.json', '105.B2C1.DT1.json', '106.B1C1.1.json', '107.B1C1.2.json', '108.B30.DT1 (к ДТ).json', '109.B30.DT2 (от ДТ).json', '110.B29.DT1.json', '111.B28.DT1.json', '112.B27.DT1.json', '113.B26.DT1.json', '114.B25.DT1.json', '115.B1C2.DT1.json', '116.B24.DT1.json', '117.B23.DT1.json', '118.B22.ASU1.json', '118.B22.DT1.json', '119.A5-A8.DT1.json', '119.B21.DT1.json', '120.A5-A8.DT1.json', '120.B21.DT1.json', '121.B20.DT1.json', '122.B19.DT1.json', '123.B18.DT1.json', '124.B17.DT1.json', '125.B16.DT1.json', '126.B15.DT1.json', '127.B14.DT1.json', '128.B13.DT1.json', '129.B12.DT1.json', '130.B11.DT1.json', '131.B10.DT1.json', '132.A3A6.DT1.json', '133.B9.DT1.json', '134.B8.DT1.json', '135.B7.DT1.json', '136.B1C9.DT1.json', '137.B1C10_DT1.json', '138.B6.DT1.json', '139.B3C9.DT2.json', '140.B3C9.DT1.json', '141.B5.DT1.json', '142.B4C11.DT1.json', '143.B4.DT1.json', '144.A4.DT1.json', '146.B1C14.DT1.json', '147.B1C14.DT2.json', '148.A1-A2.DT1.json']
    files = ['127.B14.DT1.json']
    files = os.listdir(name)
    for i in [x for x in files if x.count('.json')]:
        Print.filename = Path(name) / i
        Print.nm = i
        f = open(Print.filename)
        data = json.load(f)
        Print.dt = pd.json_normalize(data['objects'])
        Print.dt = Print.dt.explode('rows_data')
        Print.dt = pd.DataFrame(Print.dt['rows_data'].tolist())
        Print.dt = Print.dt.dropna(subset=['point_x']).reset_index(drop=True)
        Print.dt['timestamp'] =  pd.to_datetime(Print.dt['time']).astype('int64') / 10**6
        Print.protocol = {}
        for q in Print.dt.itertuples():
            Car(to_custom_object(q))
            key = q.timestamp
            if not list(Print.protocol.keys()).count(key):
                Print.protocol[key] = {'cars' : [], 'distance' : []}
        Calculate()
        Print()
        relevcar = [x.obj_id for x in Car.Allcars]
        for q in Print.dt.itertuples():
            key = q.timestamp
            if relevcar.count(q.obj_id):
                Print.protocol[key]['cars'].append({'car_id' : q.obj_id, 'point_y' : q.point_y, 'point_x' : q.point_x, 'obj_width' : q.obj_width, 'obj_length' : q.obj_length,'lane' : q.lane,'obj_speed' : q.obj_speed})

        Car.Allcars = []
        Car.Switches = 0
        CarLane.Allcars = []
        Lane.Alllanes = {}
        with open(Path(output_dir) / (str(i)[0:10] + 'dataset.txt'), 'w') as f:
            f.writelines(str(Print.protocol))
def table(output_dir):
    
    for i in list(Print.tables.keys()):
        A = set()
        for q in list(Print.tables[i].keys()):
            for p in list(Print.tables[i][q].keys()):
                if not ['N  -1', 'N  1', 'all'].count(p):
                    A.add(p)
        A = list(A)
        A.extend(['N  -1', 'N  1', 'all'])
        lines = []
        if not ['cars_speed', 'cars_distance'].count(i): 
            lines = ['time\tname\t' + '\t'.join([str(x) for x in A]) + '\n']
        keys = list(Print.tables[i].keys())
        keys.sort()
        for q in keys:
            B = [datetime.fromtimestamp(round(q)).strftime("%Y-%m-%d %H:%M"), Print.d[q][0:10]]
            if ['cars_speed', 'cars_distance'].count(i):
                for p in list(Print.tables[i][q].keys()):
                    A = [B[0], p]
                    A.extend(Print.tables[i][q][p])
                    lines.append('\t'.join([str(x) for x in A]) + '\n')
            else:
                for j in A:
                    if list(Print.tables[i][q].keys()).count(j):
                        B.append(Print.tables[i][q][j])
                    elif ['unic_cars', 'intensity_cars'].count(i):
                        B.append(0)
                    else:
                        B.append('-')
                lines.append('\t'.join([str(x) for x in B]) + '\n')

        with open(Path(output_dir) / (i + 'dataset.txt'), 'w') as f:
            f.writelines(lines)


### Run the existing solution on the supplied datasets

This processes all JSON inputs and exports derived tables and per-recording protocols to `results/`. It can take time. The supplied files in `all_tracks/` remain available for the viewer.


In [ ]:
start(DATA_DIR, OUTPUT_DIR)
table(OUTPUT_DIR)
